In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from pytorch_forecasting.data import TimeSeriesDataSet
from pytorch_forecasting.models import TemporalFusionTransformer
from torch.utils.data import DataLoader

# ---- Paths ----
PATH_H1 = r"C:\Users\admin\Desktop\Forex_algo\code\tft_ret_rich_v5.ckpt"
PATH_H4 = r"C:\Users\admin\Desktop\Forex_algo\code\tft_h4-50.44.ckpt"

DATA_H1 = r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.parquet"
DATA_H4 = r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H4.parquet"

project_root = Path(r"C:\Users\admin\Desktop\Forex_algo\code")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

device = "cuda" if torch.cuda.is_available() else "cpu"
device


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:28: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


'cuda'

In [2]:
from technicals.indicators import BollingerBands, ATR, RSI, MACD

def add_all_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # --- Classic indicators from your file ---
    df = BollingerBands(df, n=20, s=2)         # BB_MA, BB_UP, BB_LW
    df = ATR(df, n=14)                         # ATR_14
    df = RSI(df, n=14)                         # RSI_14
    df = MACD(df, n_slow=26, n_fast=12, n_signal=9)  # MACD, SIGNAL, HIST

    # Rename to cleaner names
    df = df.rename(columns={
        "BB_LW": "bb_lower",
        "BB_MA": "bb_middle",
        "BB_UP": "bb_upper",
        "ATR_14": "atr14",
        "RSI_14": "rsi",
        "MACD": "macd",
        "SIGNAL": "macd_signal",
        "HIST": "macd_hist",
    })

    # --- EMAs on mid_c ---
    df["ema_5"]   = df["mid_c"].ewm(span=5,   min_periods=5).mean()
    df["ema_20"]  = df["mid_c"].ewm(span=20,  min_periods=20).mean()
    df["ema_50"]  = df["mid_c"].ewm(span=50,  min_periods=50).mean()
    df["ema_200"] = df["mid_c"].ewm(span=200, min_periods=200).mean()

    # --- Engineered features ---
    df["momentum_oc"]        = df["mid_o"] - df["mid_c"]
    df["avg_price_hl"]       = (df["mid_h"] + df["mid_l"]) / 2.0
    df["range_hl"]           = df["mid_h"] - df["mid_l"]
    df["typical_price_ohlc"] = (df["mid_o"] + df["mid_h"] + df["mid_l"] + df["mid_c"]) / 4.0
    df["log_volume"]         = np.log(df["volume"].replace(0, np.nan))

    return df


In [3]:
def prepare_df(path: str, freq_label: str = "H1") -> pd.DataFrame:
    """
    freq_label is just for logging: "H1" or "H4".
    """
    df = pd.read_parquet(path)
    print(f"[{freq_label}] Raw shape:", df.shape)
    print(f"[{freq_label}] Columns:", list(df.columns))

    # Ensure time is datetime and sorted
    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values("time").reset_index(drop=True)

    # Compute log-return target (next bar)
    df["target_return"] = np.log(df["mid_c"].shift(-1) / df["mid_c"])

    # Add indicators & engineered features
    df = add_all_indicators(df)

    # Drop rows with any NaNs (early indicator warmup + last row with no target)
    df = df.dropna().reset_index(drop=True)

    # Add time_idx and series_id (as STRING for categoricals)
    df["time_idx"] = np.arange(len(df), dtype="int64")
    df["series_id"] = "eurusd"

    # Time-based categoricals as STR (PF expects categoricals, not numeric)
    df["hour"] = df["time"].dt.hour.astype(str)
    df["day_of_week"] = df["time"].dt.dayofweek.astype(str)

    print(f"[{freq_label}] After prep shape:", df.shape)
    print(f"[{freq_label}] Any NaNs left? {df.isna().sum().sum()}")

    return df

df_h1 = prepare_df(DATA_H1, "H1")
df_h4 = prepare_df(DATA_H4, "H4")



[H1] Raw shape: (61473, 14)
[H1] Columns: ['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h', 'bid_l', 'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c']
[H1] After prep shape: (61273, 36)
[H1] Any NaNs left? 0
[H4] Raw shape: (15375, 14)
[H4] Columns: ['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h', 'bid_l', 'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c']
[H4] After prep shape: (15175, 36)
[H4] Any NaNs left? 0


In [4]:
# Real-valued features (you can tweak, but include at least mid_c + volume + indicators)
FEATURE_COLS = [
    "mid_o", "mid_h", "mid_l", "mid_c",
    "volume",
    "rsi",
    "macd", "macd_signal", "macd_hist",
    "bb_lower", "bb_middle", "bb_upper",
    "atr14",
    "ema_5", "ema_20", "ema_50", "ema_200",
    "momentum_oc", "avg_price_hl", "range_hl",
    "typical_price_ohlc", "log_volume",
]

MAX_ENCODER_LENGTH_H1 = 96
MAX_PRED_LENGTH_H1    = 1

MAX_ENCODER_LENGTH_H4 = 48
MAX_PRED_LENGTH_H4    = 1

# ---- H1 dataset (full set, for prediction only) ----
ts_h1 = TimeSeriesDataSet(
    df_h1,
    time_idx="time_idx",
    target="target_return",
    group_ids=["series_id"],
    max_encoder_length=MAX_ENCODER_LENGTH_H1,
    max_prediction_length=MAX_PRED_LENGTH_H1,
    time_varying_unknown_reals=FEATURE_COLS,
    time_varying_known_categoricals=["hour", "day_of_week"],
    static_categoricals=["series_id"],
    target_normalizer=None,
    allow_missing_timesteps=True,
    add_relative_time_idx=True,
    add_encoder_length=True,
    add_target_scales=False,
)

# ---- H4 dataset (full set, for prediction only) ----
ts_h4 = TimeSeriesDataSet(
    df_h4,
    time_idx="time_idx",
    target="target_return",
    group_ids=["series_id"],
    max_encoder_length=MAX_ENCODER_LENGTH_H4,
    max_prediction_length=MAX_PRED_LENGTH_H4,
    time_varying_unknown_reals=FEATURE_COLS,
    time_varying_known_categoricals=["hour", "day_of_week"],
    static_categoricals=["series_id"],
    target_normalizer=None,
    allow_missing_timesteps=True,
    add_relative_time_idx=True,
    add_encoder_length=True,
    add_target_scales=False,
)

# Use built-in to_dataloader to avoid collate / NoneType issues
loader_h1 = ts_h1.to_dataloader(train=False, batch_size=256, num_workers=0)
loader_h4 = ts_h4.to_dataloader(train=False, batch_size=256, num_workers=0)

len(ts_h1), len(ts_h4)


(61177, 15127)

In [5]:
# ---- Load H1 model from Lightning checkpoint ----
tft_h1 = TemporalFusionTransformer.load_from_checkpoint(
    PATH_H1,
    map_location=device,
)
tft_h1.to(device)
tft_h1.eval()

# ---- Load H4 model from manual torch.save checkpoint ----
ckpt_h4 = torch.load(PATH_H4, map_location=device)
hparams_h4 = ckpt_h4["hyper_parameters"]

tft_h4 = TemporalFusionTransformer(**hparams_h4)
tft_h4.load_state_dict(ckpt_h4["state_dict"])
tft_h4.to(device)
tft_h4.eval()

print("H1 model loaded on", device)
print("H4 model loaded on", device)



c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
C:\Users\admin\AppData\Local\Temp\ipykernel_19568\3742389175.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In

H1 model loaded on cuda
H4 model loaded on cuda


In [ ]:
import torch

# H1 predictions
with torch.no_grad():
    preds_h1 = tft_h1.predict(
        ts_h1,
        mode="prediction",
        return_x=False,
        num_workers=0,
        batch_size=256,
    )

# H4 predictions
with torch.no_grad():
    preds_h4 = tft_h4.predict(
        ts_h4,
        mode="prediction",
        return_x=False,
        num_workers=0,
        batch_size=256,
    )

# Ensure numpy 1D arrays
if isinstance(preds_h1, torch.Tensor):
    preds_h1 = preds_h1.detach().cpu().numpy()
if isinstance(preds_h4, torch.Tensor):
    preds_h4 = preds_h4.detach().cpu().numpy()

preds_h1 = np.array(preds_h1).reshape(-1)
preds_h4 = np.array(preds_h4).reshape(-1)

# True targets aligned with the dataset
true_h1 = df_h1["target_return"].values[:len(preds_h1)]
true_h4 = df_h4["target_return"].values[:len(preds_h4)]

print("H1 preds shape:", preds_h1.shape, " | true:", true_h1.shape)
print("H4 preds shape:", preds_h4.shape, " | true:", true_h4.shape)


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 4070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoad

IndexError: index 3 is out of bounds for dimension 1 with size 1

: 